
# Demographic Transition Atlas — `dim_country` EDA and Cleaning

Этот notebook посвящён **только одной таблице**: `dim_country`.

Задача notebook:
1. спокойно и подробно разобрать, что хранится в таблице;
2. проверить качество данных;
3. аккуратно подготовить `dim_country_clean`;
4. получить понятные выводы для дальнейшего использования в:
   - `fact_indicator_value`
   - `raw_in_wpp`
   - `atlas_country_year_clean`

---

## Что мы ожидаем от `dim_country`

Это dimension table со справочной информацией по странам.

Ожидаемые колонки:
- `iso3`
- `name`
- `region`
- `income_group`
- `iso2`

### Почему эта таблица важна

Она нужна как **country reference layer**:
- нормализует страновые коды;
- помогает отделить реальные страны от агрегатов;
- даёт регион и income group для аналитики и визуализации;
- используется в joins с фактами и policy data.

---

## План notebook

Мы идём в такой последовательности:

### Часть 1. Обзор данных
- загрузка;
- shape;
- список колонок;
- первые строки;
- базовые типы;
- пропуски.

### Часть 2. Анализ качества
- уникальность `iso3`;
- дубли по `iso3`;
- дубли по `name`;
- пустые и странные значения;
- распределения по `region` и `income_group`;
- структура кодов `iso2` и `iso3`.

### Часть 3. Чистка
- trim строк;
- стандартизация регистра;
- нормализация пустых значений;
- удаление дублей по ключу;
- построение `dim_country_clean`.

### Часть 4. Подготовка к пайплайну
- сохранение clean CSV;
- подготовка списка валидных `iso3`;
- объяснение, как эта таблица будет использоваться дальше.

### Часть 5. Выводы
- что в таблице хорошо;
- что было исправлено;
- какие решения принимать в следующих notebook.

---


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path("./data_exports")
OUTPUT_DIR = Path("./notebooks/eda_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DIM_COUNTRY_PATH = DATA_DIR / "dim_country.csv"

if not DIM_COUNTRY_PATH.exists():
    raise FileNotFoundError(
        f"Файл не найден: {DIM_COUNTRY_PATH}\n"
        "Положи dim_country.csv в папку data_exports/"
    )

df_country = pd.read_csv(DIM_COUNTRY_PATH)
print("Loaded:", DIM_COUNTRY_PATH)
print("Shape:", df_country.shape)


Loaded: data_exports\dim_country.csv
Shape: (217, 5)



# 1. Первичный обзор таблицы

Сначала просто смотрим на таблицу без попытки её сразу чистить.

На этом этапе нас интересует:
- действительно ли это country dimension;
- все ли ожидаемые колонки на месте;
- нет ли явного мусора;
- похожа ли таблица на нормальный reference table.


In [2]:

df_country.head(10)


,iso3,name,region,income_group,iso2
0,ABW,Aruba,Latin America & Caribbean,High income,AW
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,AF
2,AGO,Angola,Sub-Saharan Africa,Lower middle income,AO
3,ALB,Albania,Europe & Central Asia,Upper middle income,AL
4,AND,Andorra,Europe & Central Asia,High income,AD
5,ARE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan",High income,AE
6,ARG,Argentina,Latin America & Caribbean,Upper middle income,AR
7,ARM,Armenia,Europe & Central Asia,Upper middle income,AM
8,ASM,American Samoa,East Asia & Pacific,High income,AS
9,ATG,Antigua and Barbuda,Latin America & Caribbean,High income,AG


In [3]:

print("Columns:")
print(df_country.columns.tolist())

print("\nDtypes:")
print(df_country.dtypes)


Columns:
['iso3', 'name', 'region', 'income_group', 'iso2']

Dtypes:
iso3            object
name            object
region          object
income_group    object
iso2            object
dtype: object


In [4]:

df_country.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   iso3          217 non-null    object
 1   name          217 non-null    object
 2   region        217 non-null    object
 3   income_group  217 non-null    object
 4   iso2          216 non-null    object
dtypes: object(5)
memory usage: 8.6+ KB



## Что здесь важно заметить

Для dimension table `dim_country` мы ожидаем следующее:

- число строк обычно невелико;
- колонки в основном строковые;
- `iso3` должен вести себя как естественный ключ;
- `name`, `region`, `income_group` — это справочные атрибуты, а не факты.



# 2. Missing values и базовая completeness-проверка

Сейчас посмотрим:
- сколько пропусков в каждой колонке;
- какова доля пропусков;
- есть ли критические пропуски в ключевых полях.


In [5]:

missing_summary = pd.DataFrame({
    "missing_count": df_country.isna().sum(),
    "missing_ratio": df_country.isna().mean().round(4)
}).sort_values(["missing_count", "missing_ratio"], ascending=False)

missing_summary


,missing_count,missing_ratio
iso2,1,0.0046
iso3,0,0.0000
name,0,0.0000
region,0,0.0000
income_group,0,0.0000


In [6]:

missing_plot_df = (
    missing_summary.reset_index()
    .rename(columns={"index": "column"})
)

fig = px.bar(
    missing_plot_df,
    x="column",
    y="missing_count",
    text="missing_count",
    title="dim_country — missing values by column"
)
fig.update_layout(xaxis_title="", yaxis_title="Missing count")
fig.show()



## Как интерпретировать missingness

Для `dim_country` по-настоящему критичны:
- `iso3`
- `name`

Если в них есть пропуски, строка становится проблемной как country reference.

Пропуски в:
- `region`
- `income_group`
- `iso2`

не так страшны, но их всё равно нужно учитывать:
- для графиков по регионам;
- для раскраски карты;
- для фильтров на фронте.



# 3. Нормализация строк перед анализом качества

До проверки дублей полезно сделать мягкую предварительную нормализацию:
- убрать лишние пробелы;
- привести пустые строки к `NaN`;
- стандартизировать регистр кодов.

Важно: это ещё не финальная clean-таблица.  
Это промежуточный аккуратный шаг, чтобы качество анализировалось честно.


In [7]:

country_work = df_country.copy()

def clean_string_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
    )

def blank_to_na(s: pd.Series) -> pd.Series:
    s = clean_string_series(s)
    return s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "none": pd.NA})

for col in country_work.columns:
    country_work[col] = blank_to_na(country_work[col])

if "iso3" in country_work.columns:
    country_work["iso3"] = country_work["iso3"].str.upper()

if "iso2" in country_work.columns:
    country_work["iso2"] = country_work["iso2"].str.upper()

country_work.head(10)


,iso3,name,region,income_group,iso2
0,ABW,Aruba,Latin America & Caribbean,High income,AW
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,AF
2,AGO,Angola,Sub-Saharan Africa,Lower middle income,AO
3,ALB,Albania,Europe & Central Asia,Upper middle income,AL
4,AND,Andorra,Europe & Central Asia,High income,AD
5,ARE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan",High income,AE
6,ARG,Argentina,Latin America & Caribbean,Upper middle income,AR
7,ARM,Armenia,Europe & Central Asia,Upper middle income,AM
8,ASM,American Samoa,East Asia & Pacific,High income,AS
9,ATG,Antigua and Barbuda,Latin America & Caribbean,High income,AG



# 4. Проверка ключа `iso3`

Для `dim_country` логика обычно такая:

- `iso3` — основной country key;
- он должен быть заполнен;
- он должен быть уникален;
- он должен выглядеть как 3-буквенный код.

Сейчас проверим всё это по порядку.


In [8]:

country_work["iso3"].describe()


count     217
unique    217
top       ABW
freq        1
Name: iso3, dtype: object

In [9]:

iso3_missing = country_work[country_work["iso3"].isna()].copy()
print("Rows with missing iso3:", len(iso3_missing))
iso3_missing.head(20)


Rows with missing iso3: 0


,iso3,name,region,income_group,iso2


In [10]:

iso3_dups = country_work[country_work.duplicated("iso3", keep=False)].sort_values("iso3")
print("Duplicate iso3 rows:", len(iso3_dups))
iso3_dups


Duplicate iso3 rows: 0


,iso3,name,region,income_group,iso2


In [11]:

country_work["iso3_len"] = country_work["iso3"].str.len()
iso3_len_summary = country_work["iso3_len"].value_counts(dropna=False).sort_index()
iso3_len_summary


iso3_len
3    217
Name: count, dtype: Int64

In [12]:

bad_iso3_shape = country_work[
    country_work["iso3"].notna() &
    ~country_work["iso3"].str.fullmatch(r"[A-Z]{3}", na=False)
].copy()

print("Rows with suspicious iso3 format:", len(bad_iso3_shape))
bad_iso3_shape.head(30)


Rows with suspicious iso3 format: 0


,iso3,name,region,income_group,iso2,iso3_len


Ключ iso3 надежный


# 5. Проверка `name`

Теперь смотрим на человекочитаемые названия стран.

Что проверяем:
- пропуски;
- дубли;
- потенциально странные значения;
- соответствие `name ↔ iso3`.


In [13]:

name_missing = country_work[country_work["name"].isna()].copy()
print("Rows with missing name:", len(name_missing))
name_missing.head(20)


Rows with missing name: 0


,iso3,name,region,income_group,iso2,iso3_len


In [14]:

name_dups = country_work[country_work.duplicated("name", keep=False)].sort_values("name")
print("Duplicate country names:", len(name_dups))
name_dups.head(50)


Duplicate country names: 0


,iso3,name,region,income_group,iso2,iso3_len


In [15]:

country_work["name_len"] = country_work["name"].str.len()
country_work["name_len"].describe()


count       217.0
mean     9.714286
std      5.070926
min           4.0
25%           6.0
50%           8.0
75%          11.0
max          30.0
Name: name_len, dtype: Float64

name валидно


# 6. Анализ категориальных атрибутов: `region` и `income_group`

Эти колонки особенно важны для:
- сравнительной аналитики;
- агрегаций;
- фильтров;
- раскраски графиков и карт.


In [16]:

region_summary = (
    country_work["region"]
    .value_counts(dropna=False)
    .rename_axis("region")
    .reset_index(name="n")
)

region_summary


,region,n
0,Europe & Central Asia,58
1,Sub-Saharan Africa,48
2,Latin America & Caribbean,42
3,East Asia & Pacific,37
4,"Middle East, North Africa, Afghanistan & Pakistan",23
5,South Asia,6
6,North America,3


In [17]:

fig = px.bar(
    region_summary,
    x="region",
    y="n",
    text="n",
    title="dim_country — countries by region"
)
fig.update_layout(xaxis_title="", yaxis_title="Number of rows")
fig.show()


## income_group - классификация по уровню дохода

In [18]:

income_summary = (
    country_work["income_group"]
    .value_counts(dropna=False)
    .rename_axis("income_group")
    .reset_index(name="n")
)

income_summary


,income_group,n
0,High income,86
1,Upper middle income,54
2,Lower middle income,50
3,Low income,25
4,Not classified,2


In [19]:

fig = px.bar(
    income_summary,
    x="income_group",
    y="n",
    text="n",
    title="dim_country — countries by income group"
)
fig.update_layout(xaxis_title="", yaxis_title="Number of rows")
fig.show()


In [20]:

region_income_cross = (
    country_work.groupby(["region", "income_group"], dropna=False)
    .size()
    .reset_index(name="n")
)

region_income_cross


,region,income_group,n
0,East Asia & Pacific,High income,15
1,East Asia & Pacific,Low income,1
2,East Asia & Pacific,Lower middle income,11
3,East Asia & Pacific,Upper middle income,10
4,Europe & Central Asia,High income,40
5,Europe & Central Asia,Lower middle income,3
6,Europe & Central Asia,Upper middle income,15
7,Latin America & Caribbean,High income,19
8,Latin America & Caribbean,Lower middle income,4
9,Latin America & Caribbean,Not classified,1


In [21]:

heatmap_df = region_income_cross.copy()
heatmap_df["region"] = heatmap_df["region"].astype(str)
heatmap_df["income_group"] = heatmap_df["income_group"].astype(str)

fig = px.density_heatmap(
    heatmap_df,
    x="income_group",
    y="region",
    z="n",
    histfunc="sum",
    text_auto=True,
    title="dim_country — region × income group"
)
fig.update_layout(xaxis_title="Income group", yaxis_title="Region")
fig.show()



# 7. Проверка `iso2`

`iso2` — не основной ключ, но полезный технический атрибут:
- часто нужен для внешних библиотек;
- иногда используется на фронте;
- помогает при маппинге.

Проверим:
- пропуски;
- дубли;
- формат.


In [22]:

iso2_missing = country_work[country_work["iso2"].isna()].copy()
print("Rows with missing iso2:", len(iso2_missing))
iso2_missing.head(20)


Rows with missing iso2: 1


,iso3,name,region,income_group,iso2,iso3_len,name_len
139,NAM,Namibia,Sub-Saharan Africa,Lower middle income,<NA>,3,7


одну строку подправим вручную:)

In [23]:

iso2_dups = country_work[country_work.duplicated("iso2", keep=False) & country_work["iso2"].notna()].sort_values("iso2")
print("Duplicate non-null iso2 rows:", len(iso2_dups))
iso2_dups.head(50)


Duplicate non-null iso2 rows: 0


,iso3,name,region,income_group,iso2,iso3_len,name_len


In [24]:

bad_iso2_shape = country_work[
    country_work["iso2"].notna() &
    ~country_work["iso2"].str.fullmatch(r"[A-Z]{2}", na=False)
].copy()

print("Rows with suspicious iso2 format:", len(bad_iso2_shape))
bad_iso2_shape.head(30)


Rows with suspicious iso2 format: 0


,iso3,name,region,income_group,iso2,iso3_len,name_len



# 8. Анализ уникальности справочника

Сейчас полезно собрать компактную summary-таблицу по основным quality checks.


In [25]:

quality_summary = pd.DataFrame([
    {"check": "rows_total", "value": len(country_work)},
    {"check": "unique_iso3", "value": country_work["iso3"].nunique(dropna=True)},
    {"check": "missing_iso3", "value": country_work["iso3"].isna().sum()},
    {"check": "duplicate_iso3_rows", "value": len(iso3_dups)},
    {"check": "missing_name", "value": country_work["name"].isna().sum()},
    {"check": "duplicate_name_rows", "value": len(name_dups)},
    {"check": "missing_region", "value": country_work["region"].isna().sum()},
    {"check": "missing_income_group", "value": country_work["income_group"].isna().sum()},
    {"check": "missing_iso2", "value": country_work["iso2"].isna().sum()},
    {"check": "bad_iso3_format_rows", "value": len(bad_iso3_shape)},
    {"check": "bad_iso2_format_rows", "value": len(bad_iso2_shape)},
])

quality_summary


,check,value
0,rows_total,217
1,unique_iso3,217
2,missing_iso3,0
3,duplicate_iso3_rows,0
4,missing_name,0
5,duplicate_name_rows,0
6,missing_region,0
7,missing_income_group,0
8,missing_iso2,1
9,bad_iso3_format_rows,0



# 9. Чистка: строим `dim_country_clean`

Теперь делаем уже финальную clean-таблицу.

### Что делаем конкретно

- trim строк;
- uppercase для `iso3` и `iso2`;
- пустые строки → `NaN`;
- удаляем строки без `iso3`;
- удаляем точные дубли;
- удаляем дубли по `iso3`, оставляя первую запись;
- сохраняем clean CSV.


In [32]:

dim_country_clean = country_work.copy()

# удаляем вспомогательные колонки, созданные для анализа
for col in ["iso3_len", "name_len"]:
    if col in dim_country_clean.columns:
        dim_country_clean = dim_country_clean.drop(columns=[col])

# удаляем строки без ключа
dim_country_clean = dim_country_clean.dropna(subset=["iso3"]).copy()

# удаляем точные полные дубли
dim_country_clean = dim_country_clean.drop_duplicates().copy()

# удаляем дубли по ключу iso3, если остались
dim_country_clean = dim_country_clean.drop_duplicates(subset=["iso3"], keep="first").copy()

# добиваем одну missing iso2
dim_country_clean.loc[dim_country_clean["iso3"] == "NAM", "iso2"] = "NA"

# опционально переупорядочим колонки
preferred_cols = ["iso3", "name", "region", "income_group", "iso2"]
dim_country_clean = dim_country_clean[[c for c in preferred_cols if c in dim_country_clean.columns]]

print("dim_country_clean shape:", dim_country_clean.shape)
dim_country_clean.head(10)


dim_country_clean shape: (217, 5)


,iso3,name,region,income_group,iso2
0,ABW,Aruba,Latin America & Caribbean,High income,AW
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,AF
2,AGO,Angola,Sub-Saharan Africa,Lower middle income,AO
3,ALB,Albania,Europe & Central Asia,Upper middle income,AL
4,AND,Andorra,Europe & Central Asia,High income,AD
5,ARE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan",High income,AE
6,ARG,Argentina,Latin America & Caribbean,Upper middle income,AR
7,ARM,Armenia,Europe & Central Asia,Upper middle income,AM
8,ASM,American Samoa,East Asia & Pacific,High income,AS
9,ATG,Antigua and Barbuda,Latin America & Caribbean,High income,AG



# 10. Проверка clean-версии после чистки

Теперь ещё раз убеждаемся, что clean-table выглядит так, как нам нужно.


In [33]:

print("Shape:", dim_country_clean.shape)
print("\nMissing values:")
display(dim_country_clean.isna().sum())

print("\nUnique iso3:", dim_country_clean["iso3"].nunique(dropna=True))
print("Duplicate iso3 rows after cleaning:", dim_country_clean.duplicated("iso3").sum())


Shape: (217, 5)

Missing values:


iso3            0
name            0
region          0
income_group    0
iso2            0
dtype: int64


Unique iso3: 217
Duplicate iso3 rows after cleaning: 0


In [28]:

dim_country_clean.sample(min(10, len(dim_country_clean)), random_state=42)


,iso3,name,region,income_group,iso2
205,VCT,St. Vincent and the Grenadines,Latin America & Caribbean,Upper middle income,VC
214,ZAF,South Africa,Sub-Saharan Africa,Upper middle income,ZA
138,MYS,Malaysia,East Asia & Pacific,Upper middle income,MY
177,SSD,South Sudan,Sub-Saharan Africa,Low income,SS
15,BEN,Benin,Sub-Saharan Africa,Lower middle income,BJ
111,LCA,St. Lucia,Latin America & Caribbean,Upper middle income,LC
182,SWE,Sweden,Europe & Central Asia,High income,SE
73,GMB,"Gambia, The",Sub-Saharan Africa,Low income,GM
206,VEN,"Venezuela, RB",Latin America & Caribbean,Not classified,VE
139,NAM,Namibia,Sub-Saharan Africa,Lower middle income,<NA>


In [31]:

clean_path = OUTPUT_DIR / "dim_country_clean.csv"
dim_country_clean.to_csv(clean_path, index=False)

valid_iso3 = pd.DataFrame({
    "iso3": sorted(dim_country_clean["iso3"].dropna().unique())
})

valid_iso3_path = OUTPUT_DIR / "dim_country_valid_iso3.csv"
valid_iso3.to_csv(valid_iso3_path, index=False)

print("Saved:", clean_path)
print("Saved:", valid_iso3_path)


Saved: notebooks\eda_outputs\dim_country_clean.csv
Saved: notebooks\eda_outputs\dim_country_valid_iso3.csv



# 12. Итоговые выводы по `dim_country`

Таблица чистая, готовая к последующему анализу